## `ai_query` Prompt Patterns

Two patterns for calling `ai_query()` on table data, both with **structured outputs** via `responseFormat` and parsed with `from_json`.

| | Pattern 1 | Pattern 2 |
|---|---|---|
| **Prompt source** | Inline SQL | MLflow Prompt Registry |
| **Column injection** | `\|\|` / `concat()` | `F.format_string()` |
| **Best for** | Quick, ad-hoc analysis | Reusable, versioned prompts |
| **Structured output** | Yes | Yes |
| **Model parameters** | Yes | Yes |

**`responseFormat` gotchas:**
* The DDL must have exactly **one top-level field** and it must be a `STRUCT`
* `ai_query` returns a **JSON string** regardless — use `from_json()` to parse it
* The model returns **flat JSON** (ignores the DDL wrapper), so parse with a flat schema

In [0]:
%sql
-- Sample data: product reviews
CREATE OR REPLACE TEMP VIEW product_reviews AS
SELECT * FROM VALUES
  ('R001', 'Wireless Headphones', 'Great sound quality but the battery only lasts 3 hours. Uncomfortable after long use.'),
  ('R002', 'Standing Desk',       'Sturdy build, smooth height adjustment. Motor is a bit loud though.'),
  ('R003', 'Espresso Machine',    'Makes amazing coffee. Cleaning is tedious and the water tank is too small.'),
  ('R004', 'Running Shoes',       'Lightweight and comfortable for short runs. Sole wore out after 2 months.'),
  ('R005', 'Bluetooth Speaker',   'Impressive bass for the size. Pairing drops frequently when moving between rooms.')
AS t(review_id, product_name, review_text);

SELECT * FROM product_reviews;

### Pattern 1: Inline SQL with `modelParameters` and Structured Output

The simplest approach — build the prompt directly in SQL using `||` or `concat()`, control generation with `modelParameters`, and parse the structured JSON response with `from_json`.

```
ai_query(endpoint, prompt, modelParameters => ..., responseFormat => ...)
  → JSON string → from_json() → typed columns
```

In [0]:
%sql
WITH raw AS (
  SELECT
    review_id,
    product_name,
    from_json(
      ai_query(
        'databricks-meta-llama-3-3-70b-instruct',
        concat(
          'Analyze this review for "', product_name, '".',
          ' Classify sentiment and identify the primary complaint category ',
          '(durability, usability, performance, value, or none).\n\n',
          'Review: ', review_text
        ),
        modelParameters => named_struct('temperature', 0.1, 'max_tokens', 200),
        responseFormat => 'STRUCT<result: STRUCT<sentiment: STRING, complaint_category: STRING>>'
      ),
      'sentiment STRING, complaint_category STRING'
    ) AS analysis
  FROM product_reviews
)
SELECT review_id, product_name, analysis.*
FROM raw;

### Pattern 2: MLflow Prompt Registry with Structured Output

For complex, reusable prompts — register versioned templates in the MLflow Prompt Registry, load them at runtime, render with `F.format_string()`, and parse structured output.

This has **identical parallelization performance** to inline SQL: `load_prompt()` runs once on the driver, then `format_string()` compiles to the same Catalyst plan as `concat()`.

```
Driver (once):        mlflow.genai.load_prompt() → template string
Executors (parallel): format_string() → ai_query() → from_json()
```

In [0]:
import mlflow
from pyspark.sql import functions as F
from pyspark.sql.types import StructType, StructField, StringType

UC_SCHEMA = "main.default"

# Register a complex, multi-variable prompt with version control
template = """You are a senior product analyst. For the product "{{product_name}}", \
analyze this customer review and provide:
1. A one-sentence summary
2. Sentiment (positive, negative, or mixed)
3. The primary complaint category (durability, usability, performance, value, or none)
4. A confidence level for your classification (high, medium, low)

Review: {{review_text}}"""

prompt = mlflow.genai.register_prompt(
    name=f"{UC_SCHEMA}.product_review_analysis",
    template=template,
    commit_message="Multi-field product review analysis",
    tags={"task": "analysis", "version": "v1"},
)
print(f"Registered: {prompt.name} v{prompt.version}")

# Load and render the prompt with table columns
prompt = mlflow.genai.load_prompt(f"prompts:/{UC_SCHEMA}.product_review_analysis/1")

df = spark.table("product_reviews")

template_sql = (
    prompt.template
    .replace("{{product_name}}", "%s")
    .replace("{{review_text}}", "%s")
)

# Parse structured output — flat schema matches what the model actually returns
output_schema = StructType([
    StructField("summary", StringType()),
    StructField("sentiment", StringType()),
    StructField("complaint_category", StringType()),
    StructField("confidence", StringType()),
])

result_df = (
    df.withColumn(
        "rendered_prompt",
        F.format_string(template_sql, F.col("product_name"), F.col("review_text")),
    )
    .withColumn(
        "raw_json",
        F.expr("""
            ai_query(
                'databricks-meta-llama-3-3-70b-instruct',
                rendered_prompt,
                modelParameters => named_struct('temperature', 0.1, 'max_tokens', 300),
                responseFormat => 'STRUCT<result: STRUCT<summary: STRING, sentiment: STRING, complaint_category: STRING, confidence: STRING>>'
            )
        """),
    )
    .withColumn("analysis", F.from_json(F.col("raw_json"), output_schema))
    .select("review_id", "product_name", "analysis.*")
)

display(result_df)